## Preprocessing

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_colwidth', 100)

print("Libraries loaded")

Libraries loaded


In [26]:
import os

print(os.getcwd())
print(os.listdir())

d:\HINSAFE\notebooks
['01_data_exploration.ipynb', '02_preprocessing.ipynb', '03_naive_bayes.ipynb']


In [27]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\HINSAFE")

HATECOMMENTS_PATH = PROJECT_ROOT / "datasets/raw/hatecomments.csv"
CYBERBULLYING_PATH = PROJECT_ROOT / "datasets/raw/cyberbullying_dataset.csv"
HINGLISH_PATH = PROJECT_ROOT / "datasets/raw/hinglish.csv"

OUTPUT_DIR = PROJECT_ROOT / "datasets/processed"
MERGED_FILE = OUTPUT_DIR / "hinsafe_final_dataset.csv"
TRAIN_FILE = OUTPUT_DIR / "train_split.csv"
VAL_FILE = OUTPUT_DIR / "val_split.csv"
TEST_FILE = OUTPUT_DIR / "test_split.csv"

In [28]:
# load raw datasets

df_hatecomments = pd.read_csv(HATECOMMENTS_PATH)
df_cyberbullying = pd.read_csv(CYBERBULLYING_PATH)
df_hinglish = pd.read_csv(HINGLISH_PATH)

print("Hate Comments:", df_hatecomments.shape)
print("Cyberbullying:", df_cyberbullying.shape)
print("Hinglish:", df_hinglish.shape)

Hate Comments: (1367, 2)
Cyberbullying: (25000, 3)
Hinglish: (4783, 9)


In [29]:
# satandardize column names for merging
def clean_hatecomments(df):
    """
    Standardize the Hate Comments dataset.
    Original columns: text, label (values: 'offensive', 'not offensive')
    Output columns: text, label (values: 1, 0)
    """
    df = df[["text", "label"]].copy()
    df["label"] = df["label"].map({
        "offensive": 1,
        "not offensive": 0
    })
    return df


def clean_cyberbullying(df):
    """
    Standardize the Hinglish Cyberbullying dataset.
    Original columns: ID, Text, Label (values already 0/1)
    Output columns: text, label
    """
    df = df[["Text", "Label"]].copy()
    df.columns = ["text", "label"]
    return df


def clean_hinglish(df):
    """
    Standardize the Hinglish Hate Speech dataset.
    Original columns include extra metadata we do not need.
    We keep only: text, hate_label
    Output columns: text, label
    """
    df = df[["text", "hate_label"]].copy()
    df.columns = ["text", "label"]
    return df


In [30]:
df_hatecomments_clean = clean_hatecomments(df_hatecomments)
df_cyberbullying_clean = clean_cyberbullying(df_cyberbullying)
df_hinglish_clean = clean_hinglish(df_hinglish)

print("Hate Comments cleaned:", df_hatecomments_clean.shape)
print(df_hatecomments_clean["label"].value_counts())

print("\nCyberbullying cleaned:", df_cyberbullying_clean.shape)
print(df_cyberbullying_clean["label"].value_counts())

print("\nHinglish cleaned:", df_hinglish_clean.shape)
print(df_hinglish_clean["label"].value_counts())

Hate Comments cleaned: (1367, 2)
label
1    695
0    672
Name: count, dtype: int64

Cyberbullying cleaned: (25000, 2)
label
1    15000
0    10000
Name: count, dtype: int64

Hinglish cleaned: (4783, 2)
label
0    2914
1    1869
Name: count, dtype: int64


In [31]:

for name, df in [
    ("Hate Comments", df_hatecomments_clean),
    ("Cyberbullying", df_cyberbullying_clean),
    ("Hinglish", df_hinglish_clean),
]:
    missing_labels = df["label"].isnull().sum()
    print(f"{name}: {missing_labels} rows with missing label after mapping")

Hate Comments: 0 rows with missing label after mapping
Cyberbullying: 0 rows with missing label after mapping
Hinglish: 0 rows with missing label after mapping


In [32]:
# remove duplicates from each dataset before merging
def remove_duplicates(df, name):
    """
    Removes rows where the text column is repeated.
    Keeps the first occurrence, drops the rest.
    Prints how many rows were removed.
    """
    before = df.shape[0]
    df = df.drop_duplicates(subset="text", keep="first")
    after = df.shape[0]
    removed = before - after

    print(f"{name}: removed {removed} duplicate rows ({before} -> {after})")
    return df

In [33]:
df_hatecomments_clean = remove_duplicates(df_hatecomments_clean, "Hate Comments")
df_cyberbullying_clean = remove_duplicates(df_cyberbullying_clean, "Cyberbullying")
df_hinglish_clean = remove_duplicates(df_hinglish_clean, "Hinglish")

Hate Comments: removed 30 duplicate rows (1367 -> 1337)
Cyberbullying: removed 24960 duplicate rows (25000 -> 40)
Hinglish: removed 3 duplicate rows (4783 -> 4780)


In [34]:
# Merge the cleaned datasets 
merged_df = pd.concat(
    [df_hatecomments_clean, df_cyberbullying_clean, df_hinglish_clean],
    ignore_index=True
)

print("Shape after merging, before final duplicate check:", merged_df.shape)

Shape after merging, before final duplicate check: (6157, 2)


In [35]:
# Remove any duplicate text that exists 
before = merged_df.shape[0]
merged_df = merged_df.drop_duplicates(subset="text", keep="first")
after = merged_df.shape[0]

print(f"Removed {before - after} cross-dataset duplicate rows")
print("Final merged shape:", merged_df.shape)

Removed 0 cross-dataset duplicate rows
Final merged shape: (6157, 2)


In [36]:
# Drop any rows where text or label are empty
merged_df = merged_df.dropna(subset=["text", "label"])

# Label must be an integer (0 or 1), not a float or string
merged_df["label"] = merged_df["label"].astype(int)

print("Final shape after dropping empty rows:", merged_df.shape)
print("\nFinal label distribution:")
print(merged_df["label"].value_counts())
print("\nFinal label distribution (percentage):")
print(merged_df["label"].value_counts(normalize=True).round(3) * 100)

Final shape after dropping empty rows: (6157, 2)

Final label distribution:
label
0    3594
1    2563
Name: count, dtype: int64

Final label distribution (percentage):
label
0    58.4
1    41.6
Name: proportion, dtype: float64


In [37]:
merged_df.sample(5, random_state=42)

,text,label
4890,tumhe to cricket me jeetkar bhi terrorism par hi jaana hai. https://twitter.com/AliInsafianPTI/s...,1
4651,Sahi kaha apne sir magar kya kare ye india hai yaha logo ko career banana hai jab india k andar ...,0
3153,tum logon ne indira ko mara uske baad 84hua .agar tumahra genocide aur rape karna tha to tumko 8...,0
5654,:Lage hath ye bhi bata dete ki uspe canada me rape ka case hai. Baap cycle repairer aur iska pra...,0
1586,She hate punish and bandagi iss liye.... Aur madam bolati h unbiased hu...,0


In [38]:
import os
import re


def basic_clean(text):
    """Clean text before saving and splitting."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

merged_df["text"] = merged_df["text"].apply(basic_clean)

merged_df = merged_df[
    merged_df["text"].str.strip() != ""
].copy()

merged_df = merged_df.drop_duplicates(
    subset=["text"],
    keep="first"
).reset_index(drop=True)

merged_df["label"] = merged_df["label"].astype(int)



In [39]:

# Save the cleaned merged dataset
os.makedirs(OUTPUT_DIR, exist_ok=True)

merged_df.to_csv(MERGED_FILE, index=False)

print(f"Saved cleaned dataset to: {MERGED_FILE}")
print(f"Total rows: {merged_df.shape[0]}")
print(f"Columns: {merged_df.columns.tolist()}")

Saved cleaned dataset to: D:\HINSAFE\datasets\processed\hinsafe_final_dataset.csv
Total rows: 6128
Columns: ['text', 'label']


In [40]:
# Split into Train, Validation, and Test Sets

train_val_df, test_df = train_test_split(
    merged_df,
    test_size=0.15,
    random_state=42,
    stratify=merged_df["label"]
)

val_ratio_of_remaining = 0.15 / 0.85

train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_ratio_of_remaining,
    random_state=42,
    stratify=train_val_df["label"]
)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (4288, 2)
Validation shape: (920, 2)
Test shape: (920, 2)


In [41]:

print("Train label distribution:")
print(train_df["label"].value_counts(normalize=True).round(3) * 100)

print("\nValidation label distribution:")
print(val_df["label"].value_counts(normalize=True).round(3) * 100)

print("\nTest label distribution:")
print(test_df["label"].value_counts(normalize=True).round(3) * 100)

Train label distribution:
label
0    58.4
1    41.6
Name: proportion, dtype: float64

Validation label distribution:
label
0    58.4
1    41.6
Name: proportion, dtype: float64

Test label distribution:
label
0    58.4
1    41.6
Name: proportion, dtype: float64


In [42]:
# Final safety check: confirm no text appears in more than one split
train_texts = set(train_df["text"])
val_texts = set(val_df["text"])
test_texts = set(test_df["text"])

overlap_train_val = train_texts.intersection(val_texts)
overlap_train_test = train_texts.intersection(test_texts)
overlap_val_test = val_texts.intersection(test_texts)

print("Overlap between train and validation:", len(overlap_train_val))
print("Overlap between train and test:", len(overlap_train_test))
print("Overlap between validation and test:", len(overlap_val_test))

if len(overlap_train_val) == 0 and len(overlap_train_test) == 0 and len(overlap_val_test) == 0:
    print("\nNo data leakage detected. Splits are clean.")
else:
    print("\nWarning: overlapping text found between splits. Review Section 4 and Section 5.")

Overlap between train and validation: 0
Overlap between train and test: 0
Overlap between validation and test: 0

No data leakage detected. Splits are clean.


In [43]:
# save the splits to CSV files
train_df.to_csv(TRAIN_FILE, index=False)
val_df.to_csv(VAL_FILE, index=False)
test_df.to_csv(TEST_FILE, index=False)

print("Saved:")
print(f"  {TRAIN_FILE} - {train_df.shape[0]} rows")
print(f"  {VAL_FILE} - {val_df.shape[0]} rows")
print(f"  {TEST_FILE} - {test_df.shape[0]} rows")

Saved:
  D:\HINSAFE\datasets\processed\train_split.csv - 4288 rows
  D:\HINSAFE\datasets\processed\val_split.csv - 920 rows
  D:\HINSAFE\datasets\processed\test_split.csv - 920 rows
